# NAS MLP Regression (minimal, manuel)
Objectif: comparer deux NAS simples (REINFORCE et Surrogate) sur le meme MLP.
Dataset: student_habits_performance.csv (Drive).
Split: 70/15/15.

In [58]:
# Imports
import time
import random
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from google.colab import drive

In [59]:
# Drive + path
drive.mount('/content/drive')
DATA_PATH = '/content/drive/MyDrive/Colab Notebooks/notebooks/optimisation/student_habits_performance.csv'
df = pd.read_csv(DATA_PATH)
df.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,student_id,age,gender,study_hours_per_day,social_media_hours,netflix_hours,part_time_job,attendance_percentage,sleep_hours,diet_quality,exercise_frequency,parental_education_level,internet_quality,mental_health_rating,extracurricular_participation,exam_score
0,S1000,23,Female,0.0,1.2,1.1,No,85.0,8.0,Fair,6,Master,Average,8,Yes,56.2
1,S1001,20,Female,6.9,2.8,2.3,No,97.3,4.6,Good,6,High School,Average,8,No,100.0
2,S1002,21,Male,1.4,3.1,1.3,No,94.8,8.0,Poor,1,High School,Poor,1,No,34.3
3,S1003,23,Female,1.0,3.9,1.0,No,71.0,9.2,Poor,4,Master,Good,1,Yes,26.8
4,S1004,19,Female,5.0,4.4,0.5,No,90.9,4.9,Fair,3,Master,Good,1,No,66.4


In [60]:
# Pretraitement (meme mapping que le notebook existant)
df['parental_education_level'] = df['parental_education_level'].fillna(df['parental_education_level'].mode()[0])
X = df.drop(['student_id', 'exam_score'], axis=1)
y = df['exam_score'].values

diet_map = {'Poor': 0, 'Fair': 1, 'Good': 2}
education_map = {'High School': 0, 'Bachelor': 1, 'Master': 2}
internet_map = {'Poor': 0, 'Average': 1, 'Good': 2}
gender_map = {'Female': 0, 'Male': 1, 'Other': 2}
job_map = {'No': 0, 'Yes': 1}
extra_map = {'No': 0, 'Yes': 1}

X['diet_quality'] = X['diet_quality'].map(diet_map)
X['parental_education_level'] = X['parental_education_level'].map(education_map)
X['internet_quality'] = X['internet_quality'].map(internet_map)
X['gender'] = X['gender'].map(gender_map)
X['part_time_job'] = X['part_time_job'].map(job_map)
X['extracurricular_participation'] = X['extracurricular_participation'].map(extra_map)

X = X.values.astype(np.float32)

# Split 70/15/15
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42)

# Standardisation (fit sur train)
mean = X_train.mean(axis=0)
std = X_train.std(axis=0) + 1e-8
X_train = (X_train - mean) / std
X_val = (X_val - mean) / std
X_test = (X_test - mean) / std

print(X_train.shape, X_val.shape, X_test.shape)

(700, 14) (150, 14) (150, 14)


In [61]:
# DataLoaders
def make_loader(X, y, batch_size=64, shuffle=False):
    ds = TensorDataset(torch.tensor(X), torch.tensor(y, dtype=torch.float32).view(-1, 1))
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)

train_loader = make_loader(X_train, y_train, shuffle=True)
val_loader = make_loader(X_val, y_val)
test_loader = make_loader(X_test, y_test)

In [62]:
# MLP Regressor
class MLPRegressor(nn.Module):
    def __init__(self, in_dim, n_layers, hidden_size, activation, dropout):
        super().__init__()
        act = {'relu': nn.ReLU, 'tanh': nn.Tanh, 'sigmoid': nn.Sigmoid}[activation]
        layers = []
        last = in_dim
        for _ in range(n_layers):
            layers.append(nn.Linear(last, hidden_size))
            layers.append(act())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            last = hidden_size
        layers.append(nn.Linear(last, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

def build_model(arch, in_dim):
    return MLPRegressor(
        in_dim=in_dim,
        n_layers=arch['n_layers'],
        hidden_size=arch['hidden_size'],
        activation=arch['activation'],
        dropout=arch['dropout']
    )

In [63]:
# Search space
SEARCH_SPACE = {
    'n_layers': [2, 5, 10],
    'hidden_size': [32, 64, 128, 256],
    'activation': ['relu', 'tanh', 'sigmoid'],
    'dropout': [0.0, 0.2, 0.4],
    'lr': [1e-2, 1e-3, 5e-4],
    'optimizer': ['sgd', 'adam']
}

def sample_arch(rng):
    return {k: rng.choice(v) for k, v in SEARCH_SPACE.items()}

In [64]:
def train_one_arch(arch, max_epochs=20, patience=3):
    model = build_model(arch, in_dim=X_train.shape[1])
    if arch['optimizer'] == 'sgd':
        optim = torch.optim.SGD(model.parameters(), lr=arch['lr'])
    else:
        optim = torch.optim.Adam(model.parameters(), lr=arch['lr'])
    loss_fn = nn.MSELoss()

    best_val = float('inf')
    best_state = None
    wait = 0
    start = time.time()

    for _ in range(max_epochs):
        model.train()
        for xb, yb in train_loader:
            pred = model(xb)
            loss = loss_fn(pred, yb)
            if torch.isnan(loss):
                break  # divergence détectée
            optim.zero_grad()
            loss.backward()
            optim.step()

        model.eval()
        val_preds, val_true = [], []
        with torch.no_grad():
            for xb, yb in val_loader:
                val_preds.append(model(xb).numpy())
                val_true.append(yb.numpy())
        val_preds = np.vstack(val_preds).ravel()
        val_true  = np.vstack(val_true).ravel()

        if np.any(np.isnan(val_preds)) or np.any(np.isinf(val_preds)):
            break  # arch divergente, on arrête tôt

        val_mse = mean_squared_error(val_true, val_preds)
        if val_mse < best_val:
            best_val = val_mse
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                break

    # Arch complètement divergente
    if best_state is None:
        return float('inf'), -999.0, time.time() - start

    model.load_state_dict(best_state)
    model.eval()
    test_preds, test_true = [], []
    with torch.no_grad():
        for xb, yb in test_loader:
            test_preds.append(model(xb).numpy())
            test_true.append(yb.numpy())
    test_preds = np.vstack(test_preds).ravel()
    test_true  = np.vstack(test_true).ravel()

    if np.any(np.isnan(test_preds)):
        return best_val, -999.0, time.time() - start

    return best_val, r2_score(test_true, test_preds), time.time() - start

In [65]:
# NAS 1: REINFORCE minimal
class Controller:  # distributions categorielles simples
    def __init__(self, space):
        self.space = space
        self.logits = {k: torch.zeros(len(v), requires_grad=True) for k, v in space.items()}

    def sample(self):
        arch = {}
        logprob = 0.0
        for k, vals in self.space.items():
            dist = torch.distributions.Categorical(logits=self.logits[k])
            idx = dist.sample()
            arch[k] = vals[int(idx)]
            logprob = logprob + dist.log_prob(idx)
        return arch, logprob

def run_reinforce(budget=20, lr=0.1):
    ctrl = Controller(SEARCH_SPACE)
    opt = torch.optim.SGD(ctrl.logits.values(), lr=lr)
    history = []

    for _ in range(budget):
        arch, logprob = ctrl.sample()
        val_mse, test_r2, elapsed = train_one_arch(arch)
        reward = -val_mse

        loss = -logprob * reward
        opt.zero_grad()
        loss.backward()
        opt.step()

        history.append({
            'arch': arch,
            'val_mse': val_mse,
            'test_r2': test_r2,
            'time': elapsed
        })
    return history

In [66]:
# NAS 2: Surrogate minimal
def run_surrogate(budget=20, init=6, topk=4):
    rng = random.Random(42)
    seen = []

    for _ in range(init):
        arch = sample_arch(rng)
        val_mse, test_r2, elapsed = train_one_arch(arch)
        seen.append({'arch': arch, 'val_mse': val_mse, 'test_r2': test_r2, 'time': elapsed})

    while len(seen) < budget:
        # ✅ Filtre les archs divergentes (val_mse=inf) avant d'entraîner le RF
        valid = [r for r in seen if np.isfinite(r['val_mse'])]

        if len(valid) < 2:
            # Pas assez de points valides → échantillonnage aléatoire
            arch = sample_arch(rng)
        else:
            Xs = np.array([arch_to_vec(r['arch']) for r in valid])
            ys = np.array([r['val_mse'] for r in valid])
            surrogate = RandomForestRegressor(n_estimators=50, random_state=42)
            surrogate.fit(Xs, ys)

            pool = [sample_arch(rng) for _ in range(30)]
            pool_vec = np.array([arch_to_vec(a) for a in pool])
            pred = surrogate.predict(pool_vec)
            idx = np.argsort(pred)[:topk]

            arch = pool[int(idx[0])]  # prend le meilleur prédit

        val_mse, test_r2, elapsed = train_one_arch(arch)
        seen.append({'arch': arch, 'val_mse': val_mse, 'test_r2': test_r2, 'time': elapsed})

        # Pour topk > 1, on continue la boucle while naturellement
        # Si tu veux évaluer tous les topk en une itération, remplace par :
        # for i in idx:
        #     if len(seen) >= budget: break
        #     ...

    return seen

In [67]:
# Run both (minimal budget)
reinforce_hist = run_reinforce(budget=12)
surrogate_hist = run_surrogate(budget=40, init=4, topk=2)

In [70]:
# Simple comparison
def summarize(hist, name):
    valid = [r for r in hist if np.isfinite(r['val_mse'])]
    if not valid:
        print(name, "→ aucune architecture valide")
        return
    best = min(valid, key=lambda r: r['val_mse'])
    total_time = sum(r['time'] for r in hist)
    print(name)
    print('best val mse:', best['val_mse'])
    print('test r2:', best['test_r2'])
    print('budget:', len(hist), '| valides:', len(valid))
    print('total time:', total_time)
    print('best arch:', best['arch'])

summarize(reinforce_hist, 'REINFORCE')
summarize(surrogate_hist, 'Surrogate')

REINFORCE
best val mse: 147.41917419433594
test r2: 0.4730973243713379
budget: 12 | valides: 12
total time: 5.840641975402832
best arch: {'n_layers': 5, 'hidden_size': 64, 'activation': 'relu', 'dropout': 0.4, 'lr': 0.0005, 'optimizer': 'adam'}
Surrogate
best val mse: 30.755475997924805
test r2: 0.8893450498580933
budget: 40 | valides: 36
total time: 22.85014820098877
best arch: {'n_layers': 2, 'hidden_size': 256, 'activation': 'tanh', 'dropout': 0.4, 'lr': 0.001, 'optimizer': 'sgd'}


## Rendu attendu (a remplir apres execution)
- Principe theorique de chaque algorithme (REINFORCE, Surrogate).
- Avantages / limites (stabilite, cout, efficacite echantillonnage).
- Details d'implementation (espace de recherche, entrainement, evaluation).
- Resultats: MSE val (min), R2 test (max), budget, temps total.
- Comparaison argumentee basee sur les metriques et les sorties du notebook.
